#### Import all libraries

In [56]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import timedelta
from pathlib import Path
import os

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor

#### Paths

Now we need to import our dataset. For this, we need to mention the full part of the .csv file that we are going to use.

In [3]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "final_dataset.csv"
OUTPUT_DIR = BASE_DIR / "outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

print(OUTPUT_DIR)

C:\Users\amith\Documents\GitHub\resume-projects\Predictive Forecasting of Care Load & Placement Demand\outputs


#### 1. Load data

Importing our dataset, sorting the dataset by date in ascending order

In [4]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df.sort_values(by=["Date"], inplace=True)
df.head(7)

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-402.0,NaN,NaN,NaN,3,1,0
1,2023-01-13,33.0,53.0,34.0,6622.0,415.0,33.0,53.0,34.0,6566.0,...,NaN,NaN,NaN,-381.0,NaN,NaN,NaN,4,1,0
2,2023-01-14,33.0,52.0,35.0,6677.0,394.0,33.0,53.0,34.0,6622.0,...,NaN,NaN,NaN,-359.0,NaN,NaN,NaN,5,1,1
3,2023-01-15,33.0,52.0,36.0,6733.0,373.0,33.0,52.0,35.0,6677.0,...,NaN,NaN,NaN,-337.0,NaN,NaN,NaN,6,1,1
4,2023-01-16,33.0,51.0,36.0,6788.0,352.0,33.0,52.0,36.0,6733.0,...,NaN,NaN,NaN,-316.0,NaN,NaN,NaN,0,1,0
5,2023-01-17,32.0,51.0,36.0,6844.0,332.0,33.0,51.0,36.0,6788.0,...,NaN,NaN,NaN,-296.0,NaN,NaN,NaN,1,1,0
6,2023-01-18,32.0,51.0,37.0,6900.0,311.0,32.0,51.0,36.0,6844.0,...,NaN,NaN,NaN,-274.0,-337.857143,NaN,46.063104,2,1,0


Dropping all null values and resetting index

In [5]:
df_ml = df.dropna().reset_index(drop=True)
df_ml.head(7)

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
0,2023-01-26,21.0,28.0,34.0,7522.0,211.0,20.0,22.0,41.0,7538.0,...,37.714286,7035.071429,282.714286,-177.0,-173.571429,-245.000000,38.973740,3,1,0
1,2023-01-27,22.0,34.0,26.0,7505.0,242.0,21.0,28.0,34.0,7522.0,...,37.142857,7098.142857,270.357143,-216.0,-171.428571,-233.214286,35.552108,4,1,0
2,2023-01-28,22.0,39.0,18.0,7488.0,272.0,22.0,34.0,26.0,7505.0,...,35.928571,7156.071429,261.642857,-254.0,-177.714286,-225.714286,45.893770,5,1,1
3,2023-01-29,23.0,45.0,11.0,7472.0,303.0,22.0,39.0,18.0,7488.0,...,34.142857,7208.857143,256.642857,-292.0,-192.571429,-222.500000,63.308391,6,1,1
4,2023-01-30,34.0,54.0,29.0,7743.0,196.0,23.0,45.0,11.0,7472.0,...,33.642857,7277.071429,245.500000,-167.0,-196.142857,-211.857143,60.628455,0,1,0
5,2023-01-31,26.0,36.0,36.0,7803.0,158.0,34.0,54.0,29.0,7743.0,...,33.642857,7345.571429,233.071429,-122.0,-195.285714,-199.428571,61.783801,1,1,0
6,2023-02-01,25.0,32.0,27.0,7903.0,231.0,26.0,36.0,36.0,7803.0,...,32.928571,7417.214286,227.357143,-204.0,-204.571429,-194.428571,56.579906,2,2,0


In [6]:
print("Data loaded.")
print()
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Total days:", len(df))
print()
print("Date range after lag NaNs removed:", df_ml["Date"].min().date(), "to", df_ml["Date"].max().date())
print("Days usable for feature models (after lag NaNs removed):", len(df_ml))

Data loaded.

Date range: 2023-01-12 to 2025-12-21
Total days: 1075

Date range after lag NaNs removed: 2023-01-26 to 2025-12-21
Days usable for feature models (after lag NaNs removed): 1061


#### 2. Time-based train / validation / test split
    (~67% train, ~25% val, rest test) — same idea as the notebook

In [7]:
initial_day = df["Date"].iloc[0]
final_day = df["Date"].iloc[-1]
total_days = (final_day - initial_day).days

print(total_days)

1074


In [8]:
n_train = round(total_days * 0.67)
n_val = round(total_days * 0.25)


train_end = initial_day + timedelta(days = n_train - 1)
val_end = train_end + timedelta(days = n_val - 1)

print("Training dates range from : ",initial_day,"to",train_end)
print("Validation dates ranges from : ",train_end+timedelta(1),"to",train_end)

Training dates range from :  2023-01-12 00:00:00 to 2024-12-31 00:00:00
Validation dates ranges from :  2025-01-01 00:00:00 to 2024-12-31 00:00:00


In [37]:
train_df = df[df["Date"] <= train_end].copy()
val_df = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
test_df = df[df["Date"] > val_end].copy()

train_ml = df_ml[df_ml["Date"] <= train_end].copy()
val_ml = df_ml[(df_ml["Date"] > train_end) & (df_ml["Date"] <= val_end)].copy()
test_ml = df_ml[df_ml["Date"] > val_end].copy()

In [38]:
print("\nSplit sizes:")
print("  Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))


Split sizes:
  Train: 720 | Val: 267 | Test: 88


#### Helper: evaluation metrics

In [39]:
def calc_metrics(actual, predicted):
    """Return MAE, RMSE, MAPE for one series of forecasts."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    nonzero = actual != 0
    if nonzero.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100
    return round(float(mae), 2), round(float(rmse), 2), round(float(mape), 2)

In [40]:
def accuracy_pct(mape):
    """Simple forecast accuracy % from MAPE."""
    if pd.isna(mape):
        return np.nan
    return round(max(0, 100 - mape), 2)

#### 3. Baseline models

In [41]:
def naive_forecast(series, horizon):
    """Repeat the last observed value for every future day."""
    last_value = float(series.iloc[-1])
    return np.full(horizon, last_value)

In [42]:
naive_forecast(train_df["HHS_Care"],7)

array([6430., 6430., 6430., 6430., 6430., 6430., 6430.])

In [43]:
def moving_average_forecast(series, horizon, window=7):
    """Use the recent rolling mean as the constant forecast."""
    avg = float(series.tail(window).mean())
    return np.full(horizon, avg)

In [44]:
moving_average_forecast(train_df["HHS_Care"],7)

array([6559.57142857, 6559.57142857, 6559.57142857, 6559.57142857,
       6559.57142857, 6559.57142857, 6559.57142857])

#### 4. ARIMA

In [45]:
def arima_forecast(series, horizon, order=(2, 1, 2)):
    """Fit ARIMA and forecast horizon steps ahead."""
    try:
        model = ARIMA(series.astype(float), order=order)
        fitted = model.fit()
        pred = fitted.forecast(steps=horizon)
        return np.asarray(pred, dtype=float)
    except Exception as e:
        print("  ARIMA failed:", e)
        return naive_forecast(series, horizon)

In [46]:
arima_forecast(train_df["HHS_Care"],10)

array([6471.64215182, 6498.18257942, 6491.98930537, 6460.56066979,
       6428.80028372, 6419.52555851, 6437.29208399, 6466.82957236,
       6485.84782431, 6481.68396183])

#### 5. Exponential Smoothing (Holt linear trend)

In [47]:
def exp_smoothing_forecast(series, horizon):
    """Holt linear trend (no strong weekly seasonality assumed)."""
    try:
        model = ExponentialSmoothing(
            series.astype(float),
            trend="add",
            seasonal=None,
            initialization_method="estimated"
        )
        fitted = model.fit(optimized=True)
        pred = fitted.forecast(horizon)
        return np.asarray(pred, dtype=float)
    except Exception as e:
        print("  ExpSmoothing failed:", e)
        return naive_forecast(series, horizon)

In [48]:
exp_smoothing_forecast(train_df["HHS_Care"], 7)

array([6405.88017775, 6381.76035549, 6357.64053324, 6333.52071099,
       6309.40088873, 6285.28106648, 6261.16124423])

#### 6. Machine learning models (Random Forest & Gradient Boosting)

In [49]:
FEATURE_COLS = [
    "Placed_In_CBP_lag_1", "CBP_Custody_lag_1", "CBP_To_HHS_lag_1",
    "HHS_Care_lag_1", "Discharged_From_HHS_lag_1",
    "Placed_In_CBP_lag_7", "CBP_Custody_lag_7", "CBP_To_HHS_lag_7",
    "HHS_Care_lag_7", "Discharged_From_HHS_lag_7",
    "Placed_In_CBP_lag_14", "CBP_Custody_lag_14", "CBP_To_HHS_lag_14",
    "HHS_Care_lag_14", "Discharged_From_HHS_lag_14",
    "Placed_In_CBP_rolling_avg_7", "CBP_Custody_rolling_avg_7",
    "CBP_To_HHS_rolling_avg_7", "HHS_Care_rolling_avg_7",
    "Discharged_From_HHS_rolling_avg_7",
    "Placed_In_CBP_rolling_avg_14", "CBP_Custody_rolling_avg_14",
    "CBP_To_HHS_rolling_avg_14", "HHS_Care_rolling_avg_14",
    "Discharged_From_HHS_rolling_avg_14",
    "Net_HHS_Flow", "Net_HHS_Flow_rolling_avg_7", "Net_HHS_Flow_rolling_avg_14",
    "Net_HHS_Flow_rolling_std_7",
    "Day_Of_Week", "Month", "Is_Weekend"
]

In [57]:
def train_ml_model(train_frame, target, model_type="rf"):
    """Train Random Forest or Gradient Boosting on feature columns."""
    X = train_frame[FEATURE_COLS]
    y = train_frame[target]
    if model_type == "rf":
        model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    else:
        model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)
    return model

In [58]:
def ml_predict(model, frame):
    """Predict using an already trained ML model."""
    return model.predict(frame[FEATURE_COLS])

In [62]:
val_ml

,Date,Placed_In_CBP,CBP_Custody,CBP_To_HHS,HHS_Care,Discharged_From_HHS,Placed_In_CBP_lag_1,CBP_Custody_lag_1,CBP_To_HHS_lag_1,HHS_Care_lag_1,...,CBP_To_HHS_rolling_avg_14,HHS_Care_rolling_avg_14,Discharged_From_HHS_rolling_avg_14,Net_HHS_Flow,Net_HHS_Flow_rolling_avg_7,Net_HHS_Flow_rolling_avg_14,Net_HHS_Flow_rolling_std_7,Day_Of_Week,Month,Is_Weekend
706,2025-01-01,25.0,66.0,94.0,6405.0,144.0,62.0,111.0,116.0,6430.0,...,170.000000,6563.000000,192.000000,-50.0,-37.714286,-22.000000,31.244809,2,1,0
707,2025-01-02,35.0,70.0,53.0,6292.0,170.0,25.0,66.0,94.0,6405.0,...,159.214286,6541.000000,187.142857,-117.0,-56.857143,-27.928571,33.128178,3,1,0
708,2025-01-03,38.0,71.0,57.0,6155.0,175.0,35.0,70.0,53.0,6292.0,...,147.928571,6509.357143,181.928571,-118.0,-71.428571,-34.000000,34.563262,4,1,0
709,2025-01-04,40.0,72.0,60.0,6019.0,180.0,38.0,71.0,57.0,6155.0,...,136.142857,6468.071429,176.357143,-120.0,-81.428571,-40.214286,37.344599,5,1,1
710,2025-01-05,43.0,73.0,64.0,5882.0,185.0,40.0,72.0,60.0,6019.0,...,123.857143,6417.142857,170.428571,-121.0,-86.857143,-46.571429,40.259279,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968,2025-09-20,12.0,41.0,12.0,2121.0,8.0,16.0,46.0,12.0,2109.0,...,13.571429,2068.714286,9.642857,4.0,1.000000,3.928571,5.354126,5,9,1
969,2025-09-21,8.0,35.0,13.0,2134.0,5.0,12.0,41.0,12.0,2121.0,...,13.857143,2076.428571,9.500000,8.0,1.000000,4.357143,5.354126,6,9,1
970,2025-09-22,17.0,40.0,14.0,2141.0,10.0,8.0,35.0,13.0,2134.0,...,14.500000,2084.714286,9.785714,4.0,1.571429,4.714286,5.442338,0,9,0
971,2025-09-23,11.0,36.0,10.0,2145.0,15.0,17.0,40.0,14.0,2141.0,...,14.428571,2093.428571,9.857143,-5.0,0.142857,4.571429,5.698789,1,9,0


In [68]:
model = train_ml_model(train_ml, "HHS_Care", model_type="rf")
pred = ml_predict(model, val_ml.iloc[0:7])
pred

array([6377.1 , 6380.67, 6241.07, 6077.28, 5984.54, 5847.86, 5802.22])

In [69]:
model = train_ml_model(train_ml, "HHS_Care", model_type="gb")
pred = ml_predict(model, val_ml.iloc[0:7])
pred

array([6419.6686367 , 6359.20908644, 6172.85428283, 6062.56989778,
       5931.29993869, 5869.41915424, 5811.74771306])

#### 7. Evaluate models on the TEST set